# 06 FTMO Vs Standard

This notebook compares the same strategy under two account modes:

- `standard`: research without practical account-rule limits.
- `ftmo`: applies daily-loss and max-drawdown rules from the FTMO configuration.

The goal is to see how account rules change equity, trade count, return, and drawdown.


In [ ]:
#
#

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Could not find repo root containing {marker!r} and core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:

from IPython.display import display

from core_python.strategies.combo.params import summary as strategy_summary
from core_python.strategies.combo.research_utils import (
    compare_metrics_frame,
    configure_notebook,
    plot_mode_comparison,
    show_ftmo_check,
    show_note,
    show_run_config,
)
from core_python.strategies.combo.portfolio.backtest import compare_account_modes
from core_python.strategies.combo.symbol.backtest import run_symbol_backtest

configure_notebook()
print(strategy_summary())


In [ ]:

RUN_CONFIG = {
    'symbol': 'US30',
    'portfolio_symbols': ['US30', 'US500', 'DE40', 'GOLD'],
    'initial_balance': 100_000.0,
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 30_000,
}

show_run_config('FTMO vs Standard Configuration', RUN_CONFIG)


In [ ]:
#

symbol_results = {}
for mode in ['standard', 'ftmo']:
    symbol_results[mode] = run_symbol_backtest(
        RUN_CONFIG['symbol'],
        init_eq=RUN_CONFIG['initial_balance'],
        account_mode=mode,
        date_from=RUN_CONFIG['date_from'],
        date_to=RUN_CONFIG['date_to'],
        max_bars=RUN_CONFIG['max_bars'],
    )

symbol_compare = compare_metrics_frame(symbol_results)
show_note('Symbol Mode Comparison', 'Compare KPI for the same symbol under both account modes.')
display(symbol_compare)


In [ ]:
#

mode_results = compare_account_modes(
    symbol_keys=RUN_CONFIG['portfolio_symbols'],
    initial_balance=RUN_CONFIG['initial_balance'],
    date_from=RUN_CONFIG['date_from'],
    date_to=RUN_CONFIG['date_to'],
    max_bars=RUN_CONFIG['max_bars'],
)

portfolio_compare = compare_metrics_frame(mode_results)
show_note('Portfolio Mode Comparison', 'Compare portfolio KPI between standard and ftmo modes.')
display(portfolio_compare)
show_ftmo_check(mode_results['ftmo'].metrics)


In [ ]:
#

plot_mode_comparison(symbol_results, mode_results)
